# 🚀 AIStudio Colab 全自动外挂多模态车间 (一键自愈与握手版 v2.0)
## 对应规约：AG_SOP_04 全源素材重构流水线

> **小白一键指引**：
> 你只需要点击顶部菜单 **【代码执行程序】(Runtime) ➔ 【全部运行】(Run all)**（快捷键 `Ctrl + F9`）。
> 弹窗提示时，点击 **【连接到 Google 云端硬盘】**。
> 之后**不需要输入或修改任何代码**，本脚本会自动完成环境安装、模型配置、全链路握手，并在后台自动常驻！

import os, sys, json, time
from pathlib import Path
from google.colab import drive

# 1. 自动挂载 Google Drive
drive.mount("/content/drive")

# 2. 定位统一企业中枢根目录「内容生产」
candidates = [
    Path("/content/drive/MyDrive/YOUAI/内容生产"),
    Path("/content/drive/MyDrive/内容生产"),
]
CONTENT_ROOT = None
for c in candidates:
    if c.exists():
        CONTENT_ROOT = c
        break
if CONTENT_ROOT is None:
    for p in Path("/content/drive/MyDrive").rglob("内容生产"):
        if p.is_dir():
            CONTENT_ROOT = p
            break
if CONTENT_ROOT is None:
    CONTENT_ROOT = Path("/content/drive/MyDrive/YOUAI/内容生产")
    CONTENT_ROOT.mkdir(parents=True, exist_ok=True)
STATUS_DIR = CONTENT_ROOT / ".colab_worker"
STATUS_DIR.mkdir(parents=True, exist_ok=True)
STATUS_FILE = STATUS_DIR / "handshake_status.json"

# 3. 检测 GPU
import torch
has_gpu = torch.cuda.is_available()
gpu_name = torch.cuda.get_device_name(0) if has_gpu else "No GPU (CPU)"
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3) if has_gpu else 0.0

print(f"
🖥️ 显卡状态: {gpu_name} (显存: {vram_gb:.2f} GB)")
print(f"📁 统一内容生产中枢: {CONTENT_ROOT}")

# 回写初始状态给 Antigravity
with open(STATUS_FILE, "w", encoding="utf-8") as f:
    json.dump({
        "step": 1,
        "status": "GPU_AND_DRIVE_CONNECTED",
        "gpu_name": gpu_name,
        "vram_gb": round(vram_gb, 2),
        "content_root": str(CONTENT_ROOT),
        "message": "Google Drive 内容生产中枢与 GPU 挂载成功，正在开始自动化安装开源大模型...",
        "timestamp": time.time()
    }, f, ensure_ascii=False, indent=2)

print("✅ 步骤 1 完成：已向 Google Drive 回传显卡与云盘连接信号！")


In [ ]:
import os, sys, json, time
from pathlib import Path
from google.colab import drive

# 1. 自动挂载 Google Drive
drive.mount('/content/drive')

# 2. 创建握手状态目录
STATUS_DIR = Path('/content/drive/MyDrive/AIStudio')
STATUS_DIR.mkdir(parents=True, exist_ok=True)
STATUS_FILE = STATUS_DIR / 'handshake_status.json'

# 3. 检测 GPU
import torch
has_gpu = torch.cuda.is_available()
gpu_name = torch.cuda.get_device_name(0) if has_gpu else "No GPU (CPU)"
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3) if has_gpu else 0.0

print(f"\n🖥️ 显卡状态: {gpu_name} (显存: {vram_gb:.2f} GB)")

# 回写初始状态给 Antigravity
with open(STATUS_FILE, 'w', encoding='utf-8') as f:
    json.dump({
        "step": 1,
        "status": "GPU_AND_DRIVE_CONNECTED",
        "gpu_name": gpu_name,
        "vram_gb": round(vram_gb, 2),
        "message": "Google Drive 与 GPU 挂载成功，正在开始自动化安装开源大模型...",
        "timestamp": time.time()
    }, f, ensure_ascii=False, indent=2)

print("✅ 步骤 1 完成：已向 Google Drive 回传显卡与云盘连接信号！")

### 步骤 2：全自动安装并配置开源大模型核心依赖 (faster-whisper, diffusers, F5-TTS)

In [ ]:
print("⏳ 正在一键安装开源大模型运行依赖 (大约需要 1~2 分钟，请稍候)...\n")
!pip install -q --upgrade pip
!pip install -q faster-whisper diffusers transformers accelerate soundfile sentencepiece protobuf edge-tts
!pip install -q git+https://github.com/SWivid/F5-TTS.git

# 更新状态
with open(STATUS_FILE, 'w', encoding='utf-8') as f:
    json.dump({
        "step": 2,
        "status": "DEPENDENCIES_INSTALLED",
        "message": "核心大模型依赖安装完成，正在预热加载模型引擎...",
        "timestamp": time.time()
    }, f, ensure_ascii=False, indent=2)

print("\n✅ 步骤 2 完成：所有开源大模型依赖库已全部就绪！")

def execute_manifest(manifest_path, drive_project_root=None):
    manifest_path = Path(manifest_path)
    if not manifest_path.exists():
        return
    with open(manifest_path, "r", encoding="utf-8") as f:
        manifest = json.load(f)
    project_id = manifest.get("project_id", "default_project")
    tasks = manifest.get("tasks", [])
    base_dir = manifest_path.parent if (manifest_path.parent / "02_generated").exists() else manifest_path.parent.parent
    if drive_project_root:
        candidate_dir = Path(drive_project_root) / project_id
        if candidate_dir.exists(): base_dir = candidate_dir
    print(f"
📋 正在处理项目: {project_id} (目录: {base_dir}, 共 {len(tasks)} 个任务)")
    for idx, task in enumerate(tasks, 1):
        t_type = task["type"]
        out_abs = str(base_dir / task["output_path"])
        params = task.get("params", {})
        print(f"  [{idx}/{len(tasks)}] 执行: {task['task_id']} ({t_type})")
        if t_type == "tts_narration":
            ref = params.get("ref_audio")
            if ref and not Path(ref).is_absolute(): ref = str(base_dir / ref)
            engine.generate_tts(params["text"], out_abs, ref_audio=ref, speed=params.get("speed", 1.0))
        elif t_type == "text_to_image":
            engine.generate_image(params["prompt"], out_abs, aspect_ratio=params.get("aspect_ratio", "16:9"))
        elif t_type == "asr_transcribe":
            src = params["audio_file"]
            if not Path(src).is_absolute(): src = str(base_dir / src)
            engine.transcribe(src, out_abs)
    lock_file = base_dir / "02_generated" / "manifest_completed.lock"
    lock_file.parent.mkdir(parents=True, exist_ok=True)
    with open(lock_file, "w", encoding="utf-8") as f:
        json.dump({"status": "completed", "time": time.time()}, f)
    print(f"✅ 项目 {project_id} 差额素材全部生成完成！")

def find_and_execute_manifests():
    search_dirs = [
        CONTENT_ROOT / "04_全源素材重构",
        CONTENT_ROOT / "01_真素材数字可视化",
        CONTENT_ROOT
    ]
    processed = 0
    for s_dir in search_dirs:
        if not s_dir.exists(): continue
        for p in s_dir.iterdir():
            if p.is_dir():
                mf = p / "04_编排工程" / "gap_manifest.json"
                if not mf.exists(): mf = p / "gap_manifest.json"
                lock = p / "02_generated" / "manifest_completed.lock"
                if mf.exists() and not lock.exists():
                    print(f"
🎯 发现待执行差额清单: {mf}")
                    execute_manifest(mf, drive_project_root=str(p.parent))
                    processed += 1
    return processed

print(f"🚀 统一中枢工作区已锁定: {CONTENT_ROOT}")
print("🔍 正在扫描所有待处理的差额生成任务...")
count = find_and_execute_manifests()
if count > 0:
    print(f"
🎉 成功完成 {count} 个项目的多模态差额资产生成！")
else:
    print("
💡 当前未发现待处理的新任务 (所有项目均已齐套或无需生成)。")

print("
" + "="*60)
print("⚠️ 算力节省提醒：任务已执行完毕！")
print("👉 请在顶部菜单点击：【代码执行】 -> 【断开连接并删除运行时】")
print("   以立即停止 Colab GPU 计费，保护您的每月 Compute Units！")
print("="*60)


In [ ]:
import torch
from pathlib import Path

class MultimodalEngine:
    def __init__(self):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.whisper_model = None
        self.image_pipe = None
        self.video_pipe = None
        self.f5_tts_model = None
        print(f"[Engine] 初始化成功，当前主设备: {self.device}")

    def load_whisper(self, model_size="large-v3"):
        if self.whisper_model is None:
            from faster_whisper import WhisperModel
            print(f"[ASR] 加载 faster-whisper ({model_size})...")
            compute_type = "float16" if self.device == "cuda" else "int8"
            self.whisper_model = WhisperModel(model_size, device=self.device, compute_type=compute_type)
        return self.whisper_model

    def transcribe(self, audio_path, output_json_path):
        model = self.load_whisper()
        print(f"[ASR] 开始转写: {audio_path}")
        segments, info = model.transcribe(audio_path, beam_size=5, word_timestamps=True)
        result = {"language": info.language, "duration": info.duration, "segments": []}
        for s in segments:
            words = [{"word": w.word, "start": w.start, "end": w.end, "prob": w.probability} for w in (s.words or [])]
            result["segments"].append({"id": s.id, "start": s.start, "end": s.end, "text": s.text, "words": words})
        Path(output_json_path).parent.mkdir(parents=True, exist_ok=True)
        with open(output_json_path, "w", encoding="utf-8") as f:
            json.dump(result, f, ensure_ascii=False, indent=2)
        return result

    def generate_image(self, prompt, output_path, aspect_ratio="16:9", steps=4, seed=None):
        if self.image_pipe is None:
            from diffusers import AutoPipelineForText2Image
            print(f"[T2I] 加载图像管道 (SDXL-Turbo / FLUX)...")
            self.image_pipe = AutoPipelineForText2Image.from_pretrained(
                "stabilityai/sdxl-turbo",
                torch_dtype=torch.float16,
                variant="fp16"
            ).to(self.device)
        width, height = (1024, 576) if aspect_ratio == "16:9" else ((576, 1024) if aspect_ratio == "9:16" else (1024, 1024))
        generator = torch.Generator(device=self.device).manual_seed(seed) if seed is not None else None
        image = self.image_pipe(prompt=prompt, width=width, height=height, num_inference_steps=steps, generator=generator).images[0]
        Path(output_path).parent.mkdir(parents=True, exist_ok=True)
        image.save(output_path)
        return output_path

    def generate_tts(self, text, output_path, ref_audio=None, speed=1.0):
        Path(output_path).parent.mkdir(parents=True, exist_ok=True)
        try:
            from f5_tts.api import F5TTS
            if self.f5_tts_model is None:
                print("[TTS] 加载 F5-TTS 声音克隆模型...")
                self.f5_tts_model = F5TTS()
            self.f5_tts_model.infer(ref_file=ref_audio, ref_text="", gen_text=text, speed=speed, file_wave=output_path)
        except Exception as e:
            print(f"[TTS] F5-TTS 备选轻量合成: {e}")
            import subprocess
            subprocess.run(["edge-tts", "--text", text, "--write-media", output_path, "--voice", "zh-CN-YunxiNeural"], check=True)
        return output_path

engine = MultimodalEngine()

# 更新状态
with open(STATUS_FILE, 'w', encoding='utf-8') as f:
    json.dump({
        "step": 3,
        "status": "ALL_MODELS_READY",
        "message": "🎉 所有开源大模型全部部署成功，多模态引擎就绪！",
        "timestamp": time.time()
    }, f, ensure_ascii=False, indent=2)

print("\n✨ 步骤 3 完成：多模态大模型引擎已全部装载就绪！")

### 步骤 4：自动执行全链路握手测试 (合成第一声握手语音并回传 Drive)

In [ ]:
HANDSHAKE_WAV = STATUS_DIR / 'projects' / 'test_handshake' / '02_generated' / 'voices' / 'handshake_ok.wav'
HANDSHAKE_WAV.parent.mkdir(parents=True, exist_ok=True)

print("🎙️ 正在进行端到端握手测试：合成第一条 AI 语音...")
engine.generate_tts(
    text="恭喜！Antigravity 与 Google Colab 外挂多模态车间全链路握手成功！模型配置完毕！",
    output_path=str(HANDSHAKE_WAV)
)

# 更新最终握手状态
with open(STATUS_FILE, 'w', encoding='utf-8') as f:
    json.dump({
        "step": 4,
        "status": "HANDSHAKE_SUCCESSFUL",
        "gpu": gpu_name,
        "vram_gb": round(vram_gb, 2),
        "handshake_audio": str(HANDSHAKE_WAV),
        "message": "🎊 全链路握手完全成功！所有模型就绪，已随时准备好接收内容生产任务！",
        "timestamp": time.time()
    }, f, ensure_ascii=False, indent=2)

print(f"\n🎊 握手语音生成完毕: {HANDSHAKE_WAV}")
print("🎉 全链路握手完全成功！Antigravity 已经在后台检测到此信号！")

### 步骤 5：进入自动常驻监听循环 (随时接收 Antigravity 派发的视频生产任务)

In [ ]:
def execute_manifest(manifest_path, drive_project_root="/content/drive/MyDrive/AIStudio"):
    manifest_path = Path(manifest_path)
    if not manifest_path.exists():
        return
    with open(manifest_path, "r", encoding="utf-8") as f:
        manifest = json.load(f)
    project_id = manifest.get("project_id", "default_project")
    tasks = manifest.get("tasks", [])
    base_dir = Path(drive_project_root) / project_id
    print(f"\n📋 正在处理项目: {project_id} (共 {len(tasks)} 个任务)")
    for idx, task in enumerate(tasks, 1):
        t_type = task["type"]
        out_abs = str(base_dir / task["output_path"])
        params = task.get("params", {})
        print(f"  [{idx}/{len(tasks)}] 执行: {task['task_id']} ({t_type})")
        if t_type == "tts_narration":
            ref = params.get("ref_audio")
            if ref and not Path(ref).is_absolute(): ref = str(base_dir / ref)
            engine.generate_tts(params["text"], out_abs, ref_audio=ref, speed=params.get("speed", 1.0))
        elif t_type == "text_to_image":
            engine.generate_image(params["prompt"], out_abs, aspect_ratio=params.get("aspect_ratio", "16:9"))
        elif t_type == "asr_transcribe":
            src = params["audio_file"]
            if not Path(src).is_absolute(): src = str(base_dir / src)
            engine.transcribe(src, out_abs)
    lock_file = base_dir / "02_generated" / "manifest_completed.lock"
    lock_file.parent.mkdir(parents=True, exist_ok=True)
    with open(lock_file, "w", encoding="utf-8") as f:
        json.dump({"status": "completed", "time": time.time()}, f)
    print(f"✅ 项目 {project_id} 差额素材全部生成完成！")

print("👀 开始自动常驻监听任务...")
watch_dir = Path("/content/drive/MyDrive/AIStudio/projects")
while True:
    try:
        for p in watch_dir.iterdir():
            if p.is_dir():
                mf = p / "04_编排工程" / "gap_manifest.json"
                if not mf.exists(): mf = p / "gap_manifest.json"
                lock = p / "02_generated" / "manifest_completed.lock"
                if mf.exists() and not lock.exists():
                    execute_manifest(mf, drive_project_root=str(watch_dir))
        time.sleep(15)
    except KeyboardInterrupt:
        print("\n已主动退出监听。")
        break
    except Exception as e:
        time.sleep(15)
